In [2]:
from typing import Annotated, TypedDict
from langchain_ollama import ChatOllama
from langchain_core.messages import SystemMessage, HumanMessage, BaseMessage
from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from langchain_core.tools import tool
from src.retrieval.embedders import SentenceTransformerEmbedder
from src.retrieval.store import ElasticSearchVectorStore

embedder = SentenceTransformerEmbedder()
cve_store = ElasticSearchVectorStore(index_name='cve_index')
mitre_store = ElasticSearchVectorStore(index_name='mitre_attack')

@tool
def query_cve(query: str) -> str:
    '''Tool to query CVEs from the vector store. Useful for finding vulnerabilities related to specific software, attack techniques, or CVE IDs.'''
    vector = embedder.embed_query(query)
    results = cve_store.search(vector, top_k=3)
    return '\n\n'.join([f"{res['id']}: {res['description']}" for res in results])

@tool
def query_mitre(query: str) -> str:
    '''Tool to query MITRE techniques from the vector store. Useful for finding information about specific attack techniques.'''
    vector = embedder.embed_query(query)
    results = mitre_store.search(vector, top_k=3)
    return '\n\n'.join([f"{res['id']}: {res['description']}" for res in results])

c:\Users\omarinf\Documents\TFM\alert_correlation\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 12901.70it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [9]:
class AgentState(TypedDict):
    messages: Annotated[BaseMessage, add_messages]

llm = ChatOllama(model='llama3.1:8b')

available_tools = [query_cve, query_mitre]
llm_with_tools = llm.bind_tools(available_tools)

SYSTEM_PROMPT = '''
You are a SOC expert analyst. Your work is to analyze security alerts, logs and threat intelligence to identify potential security incidents and recommend appropriate responses.
Your goal is to analyze security alerts and enrich them with relevant information from CVE and MITRE databases.
YOU HAVE TOOLS AVAILABLE TO YOU, USE THEM WHEN NECESSARY TO COMPLETE YOUR TASKS.

GOLD RULE:
If user provides you with an alert, you MUST analyze it and enrich it with relevant information from CVE and MITRE databases. 
You MUST use the tools available to you to retrieve this information. You MUST NOT provide a final answer without using the tools to retrieve relevant information.

ROUTING RULES:
1. Behaviors, commands, processes or malware -> USE query_mitre TOOL
2. Software versions, scan results or vulnerability identifiers -> USE query_cve TOOL
3. Vulnerabilities + Post-explotation behaviors -> USE BOTH TOOLS
4. If you need to retrieve information to complete your tasks, ALWAYS USE THE TOOLS.
5. If do not know which tool to use, USE BOTH TOOLS.

FINAL ANSWER RULES:
ONLY when you have retrieved relevant information from the tools, you can provide a final answer.
Use EXACTLY this structure in Markdown format for your final answer:
### Incident Summary
[Summary]
### Intelligence Correlation
[CVEs and MITRE explanations. If both, explain relationship between them]
### Priority and Severity
[Your assessment of the priority and severity of the incident, based on the information you have]
### Recommended Response
[Your recommended response to the incident, including any immediate actions that should be taken and any further investigation that may be needed]
Answer in a structured, professional and concise way.
'''

def agent_node(state: AgentState):
    '''Agent node that analyze the alert and decides if use tools or writes a final answer.'''
    messages = state['messages']
    
    if not isinstance(messages[0], SystemMessage):
        messages = [SystemMessage(content=SYSTEM_PROMPT)] + messages

    response = llm_with_tools.invoke(messages)
    print(f'DEBUG: Tool calls: {response.tool_calls}')
    return {'messages': [response]}


def use_tool_decision(state: AgentState):
    '''Decision node that determines whether the agent should use a tool or not.'''
    last_message = state['messages'][-1]
    
    if hasattr(last_message, 'tool_calls') and len(last_message.tool_calls) > 0:
        return 'action'
    
    return 'end'

In [10]:
workflow = StateGraph(AgentState)
workflow.add_node('agent', agent_node)
workflow.add_node('action', ToolNode(available_tools))
workflow.set_entry_point('agent')
workflow.add_conditional_edges(
    'agent',
    use_tool_decision,
    {
        'action': 'action',
        'end': END
    }
)

workflow.add_edge('action', 'agent')
app = workflow.compile()

In [3]:
user_input = input("\nUser: ")

inputs = {'messages': [HumanMessage(content=user_input)]}
for event in app.stream(inputs, stream_mode='values'):
    last_message = event['messages'][-1]
    if hasattr(last_message, 'content') and last_message.content and not hasattr(last_message, 'tool_calls'):
        print(f"Agent: {last_message.content}")

Agent: hola
Agent: CVE-2025-59832: Horilla is a free and open source Human Resource Management System (HRMS). Prior to version 1.4.0, there is a stored XSS vulnerability in the ticket comment editor. A low-privilege authenticated user could run arbitrary JavaScript in an admin’s browser, exfiltrate the admin’s cookies/CSRF token, and hijack their session. This issue has been patched in version 1.4.0.

CVE-2026-40867: Horilla is a free and open source Human Resource Management System (HRMS). In 1.5.0, a broken access control vulnerability in the helpdesk attachment viewer allows any authenticated user to view attachments from other tickets by changing the attachment ID. This can expose sensitive support files and internal documents across unrelated users or teams.

CVE-2025-48869: Horilla is a free and open source Human Resource Management System (HRMS). Unauthenticated users can access uploaded resume files in Horilla 1.3.0 by directly guessing or predicting file URLs. These files are 

In [ ]:
# Invoke
user_input = 'EDR alert: Detected CVE-2021-44228 exploitation attempt on web server. '

inputs = {'messages': [HumanMessage(content=user_input)]}
messages = app.invoke(inputs)
for m in messages["messages"]:
    m.pretty_print()

DEBUG: Tool calls: []
================================ Human Message =================================

EDR alert: Detected CVE-2021-44228 exploitation attempt on web server. 
================================== Ai Message ==================================

### Incident Summary
EDR alert indicates a possible exploitation attempt of CVE-2021-44228 on a web server.

### Intelligence Correlation
We will use the `query_cve` tool to gather information about the identified vulnerability.
The MITRE database may also provide additional context or related attack techniques that an attacker could utilize.

### Priority and Severity
**Priority: High**
**Severity: Critical**

### Recommended Response
Firstly, we recommend investigating the web server's logs for any suspicious activity related to this exploitation attempt. 
Next, ensure that all software on the affected system is up-to-date, especially Log4j version 2.
Further investigation should be conducted to identify potential entry points and

In [11]:
for event in app.stream(inputs, stream_mode='updates'):
    for node_name, node_state in event.items():
        print(f'\nNODO: [{node_name}]')

        last_message = node_state['messages'][-1]

        if hasattr(last_message, 'tool_calls') and last_message.tool_calls:
            for tool_call in last_message.tool_calls:
                print(f"Tool call: {tool_call}")

        elif getattr(last_message, 'type', '') == 'tool':
            resp = last_message.content[:150].replace('\n', ' ')
            print(f"Tool response [{last_message.name}]: {resp}...")

        elif getattr(last_message, 'type', '') == 'ai':
            print(f"AI: {last_message.content}")
            


DEBUG: Tool calls: []

NODO: [agent]
AI: ### Incident Summary
EDR alert detected CVE-2021-44228 exploitation attempt on a web server.

### Intelligence Correlation
The detected CVE is related to a known vulnerability in Apache Log4j, tracked as CVE-2021-44228 by the Common Vulnerabilities and Exposures (CVE) database. This vulnerability allows an attacker to execute arbitrary code on the affected system through a crafted log message. According to MITRE, this vulnerability is associated with the attack technique T1190 - Exploit Use of Out-of-Date Software.

### Priority and Severity
High priority, High severity. The exploitation attempt indicates potential compromise of the web server, which could lead to data theft or other malicious activities.

### Recommended Response
Immediately investigate the web server for signs of compromise and take remediation steps to patch the vulnerability. This may involve applying a security update or reconfiguring the Log4j version to a secure one. Moni

## Prueba sin tools

In [23]:
import os
from typing import Annotated, TypedDict, List
from pydantic import BaseModel, Field

from langgraph.graph import StateGraph, END
from langchain_core.messages import SystemMessage, HumanMessage, BaseMessage
from langchain_ollama import ChatOllama


from src.retrieval.embedders import SentenceTransformerEmbedder
from src.retrieval.store import ElasticSearchVectorStore

embedder = SentenceTransformerEmbedder()
cve_store = ElasticSearchVectorStore(index_name='cve_index')
mitre_store = ElasticSearchVectorStore(index_name='mitre_attack')
# llm = ChatOllama(model='llama3.1:8b')

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4143.97it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [86]:
class AlertClasification(BaseModel):
    mitre_search: bool = Field(description="True if the alert contains behaviors, commands, processes, tactics or malware. ONLY set to False if you are 100% sure that the alert does not contain any information that can be used to search in MITRE database")
    mitre_keywords: str = Field(description="The exact query to search in MITRE database. Empty if mitre_search is False")
    mitre_description: str = Field(description="Detailed description of the alert related to MITRE information. DO NOT try to identify the Technique associated, TRY to explain the context to search in the vector database. Empty if mitre_search is False")
    cve_search: bool = Field(description="True if the alert contains software versions, scan results or vulnerability identifiers. ONLY set to False if you are 100% sure that the alert does not contain any information that can be used to search in CVE database")
    cve_keywords: str = Field(description="The exact query to search in CVE database. Empty if cve_search is False")
    cve_description: str = Field(description="Detailed description of the alert related to CVE information. DO NOT try to identify the CVE associated, TRY to explain the context to search in the vector database. Empty if cve_search is False")

class AgentState(TypedDict):
    original_alert: str
    classification: AlertClasification
    mitre_data: Annotated[str, Field(description="Relevant MITRE information retrieved")]
    cve_data: Annotated[str, Field(description="Relevant CVE information retrieved")]
    final_report: Annotated[str, Field(description="Final report generated by the agent")]


def classification_node(state: AgentState) -> AgentState:
    '''Node that classifies the alert and extracts relevant keywords for MITRE and CVE searches.'''
    prompt = f'''
    You are a SOC analyst. Your task is to analyze the following security alert and determine if it contains information that can be used to search in MITRE and CVE databases.

    Alert: {state['original_alert']}

    For MITRE, you are looking for behaviors, commands, processes, tactics or malware mentioned in the alert. If you find any, set mitre_search to True and extract the relevant keywords for searching in MITRE database.

    For CVE, you are looking for software versions, scan results or vulnerability identifiers mentioned in the alert. If you find any, set cve_search to True and extract the relevant keywords for searching in CVE database.
    
    EXAMPLES:
    - Alert: "Multiple failed login attempts in RDP" mitre_search: True (brute force), cve_search: False (no vulnerabilities mentioned)
    - Alert: "WINRAR old version detected that allows remote code execution" mitre_search: False, cve_search: True (WINRAR vulnerability)
    - Alert: "Mimikatz process detected running in memory" mitre_search: True (credential dumping), cve_search: False (no vulnerabilities mentioned)
    - Alert: "Proxyshell exploitation (CVE-2021-34473) followed by webshell detected" mitre_search: True (Web Shell), cve_search: True (Proxyshell vulnerability)
    
    At the slightest suspicion of malicious behavior, you MUST set mitre_search to True, and if there is any mention of exploitable software, vulnerabilities or patches, you MUST set cve_search to True.
    '''
    structured_llm = llm.with_structured_output(AlertClasification)
    classification = structured_llm.invoke([SystemMessage(content=prompt),
                                            HumanMessage(content=f'The alert to analyze is: {state["original_alert"]}')])
    return {'classification': classification}


def mitre_search_node(state: AgentState) -> AgentState:
    '''Node that performs MITRE search if mitre_search is True.'''

    if not state['classification'].mitre_search:
        return {'mitre_data': 'No relevant MITRE information found in the alert.'}
    
    query = state['classification'].mitre_description
    query_vector = embedder.embed_query(query)
    results = mitre_store.search(query_vector, top_k=3)

    results_text = 'MITRE results:\n' + '\n'.join([f"Technique ID: {r['technique_id']}\n"
                    f"Name: {r['name']}\n"
                    f"Description: {r['description']}\n"
                    f"Tactics: {r['tactics']}\n"
                    f"Platforms: {r['platforms']}\n"
                    "-----------------" for r in results])
    return {'mitre_data': results_text}

def cve_search_node(state: AgentState) -> AgentState:
    '''Node that performs CVE search if cve_search is True.'''

    if not state['classification'].cve_search:
        return {'cve_data': 'No relevant CVE information found in the alert.'}

    query = state['classification'].cve_description
    query_vector = embedder.embed_query(query)
    results = cve_store.search(query_vector, top_k=3)

    results_text = 'CVE results:\n' + '\n'.join([f"CVE ID: {r['id']}\n"
                    f"Title: {r['title']}\n"
                    f"Description: {r['description']}\n"
                    f"Published Date: {r['published_date']}\n"
                    f"CVSS: {r['cvss']}\n"
                    f"Affected Products: {r['versions']}\n"
                    f"Mitigations: {r['mitigations']}\n"
                    f"SSVC: {r['ssvc']}\n"
                    f"References: {r['references']}\n"
                    f"In KEV: {r['in_kev']}\n"
                    "-----------------" for r in results])
    return {'cve_data': results_text}

def final_report_node(state: AgentState) -> AgentState:
    '''Node that generates a final report based on the original alert, the classification, and the retrieved MITRE and CVE information.'''
    prompt = f'''
    You are a SOC analyst. Your task is to generate a final report based on the information about the alert, the relevant MITRE techniques and CVE vulnerabilities that have been identified.
    Generate a concise report in Markdown format summarizing the incident, correlating the MITRE and CVE information with the original alert, and providing an assessment of the priority and severity of the incident, as well as recommended response actions.
    '''

    user_input = f'''
    Original Alert: {state['original_alert']}
    Classification: {state['classification']}
    MITRE Information: {state['mitre_data']}
    CVE Information: {state['cve_data']}'''

    response = llm.invoke([SystemMessage(content=prompt), HumanMessage(content=user_input)])
    # Escribir en un archivo de texto el informe final
    with open('final_report.md', 'w') as f:
        f.write(response.content[0]['text'])
    
    return {'final_report': response.content[0]}


def conditional_router(state: AgentState) -> List[str]:
    '''Node to determine which nodes to execute based on the classification results.'''
    classification = state['classification']
    nodes_to_execute = []
    if classification.mitre_search:
        nodes_to_execute.append('mitre_search_node')
    if classification.cve_search:
        nodes_to_execute.append('cve_search_node')


    if not nodes_to_execute:
        nodes_to_execute.append('final_report_node')

    return nodes_to_execute

In [37]:
q = "The alert explicitly identifies CVE-2021-44228, also known as Log4Shell, which is a critical vulnerability in the Apache Log4j 2 logging library.'"
q_vector = embedder.embed_query(q)
results = cve_store.search(q_vector, top_k=3)
results_text = 'CVE results:\n' + '\n'.join([f"CVE ID: {r['id']}\n"
                    f"Title: {r['title']}\n"
                    f"Description: {r['description']}\n"
                    f"Published Date: {r['published_date']}\n"
                    f"CVSS: {r['cvss']}\n"
                    f"Affected Products: {r['versions']}\n"
                    f"Mitigations: {r['mitigations']}\n"
                    f"SSVC: {r['ssvc']}\n"
                    f"References: {r['references']}\n"
                    f"In KEV: {r['in_kev']}\n"
                    "-----------------" for r in results])

print(results_text)

CVE results:
CVE ID: CVE-2021-45046
Title: Apache Log4j2 Deserialization of Untrusted Data Vulnerability
Description: It was found that the fix to address CVE-2021-44228 in Apache Log4j 2.15.0 was incomplete in certain non-default configurations. This could allows attackers with control over Thread Context Map (MDC) input data when the logging configuration uses a non-default Pattern Layout with either a Context Lookup (for example, $${ctx:loginId}) or a Thread Context Map pattern (%X, %mdc, or %MDC) to craft malicious input data using a JNDI Lookup pattern resulting in an information leak and remote code execution in some environments and local code execution in all environments. Log4j 2.16.0 (Java 8) and 2.12.2 (Java 7) fix this issue by removing support for message lookup patterns and disabling JNDI functionality by default.
Published Date: 2021-12-14T16:55:09.000Z
CVSS: {'score': 9, 'severity': 'CRITICAL', 'vector': 'CVSS:3.1/AV:N/AC:H/PR:N/UI:N/S:C/C:H/I:H/A:H'}
Affected Products:

In [3]:
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv

load_dotenv()

API_KEY = os.getenv('GEMINI_TOKEN')

llm = ChatGoogleGenerativeAI(model='gemini-3.1-flash-lite', api_key=API_KEY, temperature=0.2)

In [87]:
workflow = StateGraph(AgentState)
workflow.add_node('classification', classification_node)
workflow.add_node('mitre_search_node', mitre_search_node)
workflow.add_node('cve_search_node', cve_search_node)
workflow.add_node('final_report_node', final_report_node)

workflow.set_entry_point('classification')

workflow.add_edge('classification', 'mitre_search_node')
workflow.add_edge('classification', 'cve_search_node')

# workflow.add_conditional_edges('classification', conditional_router)

workflow.add_edge(['mitre_search_node', 'cve_search_node'], 'final_report_node')

workflow.add_edge('classification', END)

app = workflow.compile()

In [88]:
user_input = 'EDR alert: Detected exploitation attempt on web server. '

inputs = {'original_alert': user_input}
for event in app.stream(inputs, stream_mode='updates'):
    for node_name, node_state in event.items():
        print(f'\nNODO: [{node_name}]')

        if 'classification' in node_state:
            print(f"Classification: {node_state['classification']}")

        if 'mitre_data' in node_state:
            print(f"MITRE Data: {node_state['mitre_data'][:200]}...")

        if 'cve_data' in node_state:
            print(f"CVE Data: {node_state['cve_data'][:200]}...")

        if 'final_report' in node_state:
            print(f"Final Report: {node_state['final_report']}")


NODO: [classification]
Classification: mitre_search=True mitre_keywords='exploitation attempt web server' mitre_description='The alert indicates an active attempt to exploit a web server, which relates to tactics such as Initial Access, Execution, or Persistence.' cve_search=False cve_keywords='' cve_description=''

NODO: [cve_search_node]
CVE Data: No relevant CVE information found in the alert....

NODO: [mitre_search_node]
MITRE Data: MITRE results:
Technique ID: T1204
Name: User Execution
Description: An adversary may rely upon specific actions by a user in order to gain execution. Users may be subjected to social engineering to g...

NODO: [final_report_node]
Final Report: {'type': 'text', 'text': "# Incident Report: Web Server Exploitation Attempt\n\n## 1. Executive Summary\nThe EDR system triggered an alert regarding an active exploitation attempt targeting a web server. Analysis suggests an adversary is attempting to gain unauthorized access or execute code, likely leveraging 